# 4. Multi-Tenant Vector Databasing with Milvus for RAG and Fine-tuning with GPU/QAIC (Instructor: Mohammad Sada)

The tutorial focuses on NRP-managed Milvus as a multi-tenant vector database service. Participants are guided through accessing Milvus with their credentials, defining collections, and inserting embeddings for semantic search. Using either GPU or QAIC resources, attendees will integrate Milvus into RAG or fine-tuning workflows.

## Milvus on NRP

NRP runs **Milvus** as a managed service, so you don’t install the operator or create the instance yourself. The gRPC endpoint is **`milvus.nrp-nautilus.io:50051`**. To get access, use the [Milvus password page](https://nrp.ai/milvus) and click “Get milvus password”; the secure link is emailed to you. You must be in a [namespace/group with Milvus enabled](https://nrp.ai/namespaces). Group names may use letters, numbers, and dashes; any dashes are converted to underscores in the Milvus database name. From there you can [define collections](https://milvus.io/docs/manage-collections.md), run [vector search](https://milvus.io/docs/single-vector-search.md), or use the [Attu GUI](https://milvus.io/docs/quickstart_with_attu.md). Full reference: [NRP vector database](https://nrp.ai/documentation/userdocs/vector-database/).

## RAG example using Milvus
### Credit: Mohammad Sada, SDSC

**Prerequisite:** Get your Milvus password and ensure your group has Milvus enabled (see “Milvus on NRP” above) before starting the pod.

This example demonstrates RAG using Milvus as the vector database. In `yamls/milvus-rag.yaml`, **replace `<username>`** with your username so the pod name `tutorial-<username>-vectordb` is unique. Start up the pod (namespace `nrp-training-k8s`; create it with `kubectl create namespace nrp-training-k8s` if needed):
```
kubectl apply -n nrp-training-k8s -f yamls/milvus-rag.yaml
```

In [ ]:
# Run this cell once to apply subtle formatting for easier reading (optional)
from IPython.display import HTML, display
display(HTML("""
<style>
  .jp-MarkdownCell .jp-Cell-inputWrapper,
  .text_cell .input_area { background: linear-gradient(90deg, #f8f9fa 0%, transparent 12px); border-left: 3px solid #dee2e6; padding: 0.6em 0 0.6em 1em; margin: 0.4em 0; border-radius: 0 4px 4px 0; }
  .jp-MarkdownCell h1, .text_cell_render h1 { color: #212529; font-weight: 600; border-bottom: 1px solid #e9ecef; padding-bottom: 0.35em; margin-top: 0; }
  .jp-MarkdownCell h2, .text_cell_render h2 { color: #0d6efd; font-weight: 600; margin-top: 1.1em; margin-bottom: 0.4em; }
  .jp-MarkdownCell h3, .text_cell_render h3 { color: #198754; font-weight: 600; margin-top: 0.9em; }
  .jp-MarkdownCell p, .jp-MarkdownCell li, .text_cell_render p, .text_cell_render li { line-height: 1.6; color: #374151; }
  .jp-MarkdownCell code, .text_cell_render code { background: #f1f3f4; padding: 0.2em 0.4em; border-radius: 4px; font-size: 0.9em; }
  .jp-MarkdownCell strong, .text_cell_render strong { color: #1f2937; }
  .jp-MarkdownCell hr, .text_cell_render hr { border-color: #e5e7eb; margin: 1.2em 0; }
</style>
"""))

Apply the Milvus RAG pod:

In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/milvus-rag.yaml

Watch the logs and make sure you wait till the installs are done:

```
kubectl logs -n nrp-training-k8s tutorial-<username>-vectordb
```
Replace `<username>` with the name you used in the YAML.

In [ ]:
!kubectl logs -n nrp-training-k8s tutorial-<username>-vectordb

The pod automatically installs Python dependencies, Ollama, starts the Ollama server, and downloads the mistral model. Download the example script into the pod once it is running (example repo: [groundsada/nrp-milvus-example](https://github.com/groundsada/nrp-milvus-example)):
```
kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# then inside the pod:
cd /scratch
wget https://raw.githubusercontent.com/groundsada/nrp-milvus-example/refs/heads/main/milvus-example.py
```

In [ ]:
!kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# In the next cell or inside the pod: cd /scratch && wget https://raw.githubusercontent.com/groundsada/nrp-milvus-example/refs/heads/main/milvus-example.py

Once the installation is complete (check logs), you can run the example. This example connects to the `sc25_milvus` database using credentials from the Kubernetes secret `sc25-milvus-credentials`. The collection name is `simple_rag_example`. To use a different collection, edit `milvus-example.py` and change `collection_name="simple_rag_example"` (there are two places where the collection name is set).

Run the script inside the pod (see next cell). The script uses environment variables for the Milvus connection.

For all other aspects, the script uses environment variables for Milvus connection, so no manual editing is needed.

```
kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# then inside the pod:
cd /scratch
python3 milvus-example.py
```

In [ ]:
!kubectl exec -it -n nrp-training-k8s tutorial-<username>-vectordb -- /bin/bash
# Then run: cd /scratch && python3 milvus-example.py

---
## Embedding Generation: NVIDIA vs Qualcomm Backends

The embedding and RAG workflow above uses an **NVIDIA GPU** pod with Ollama. The same workflow can be adapted for **Qualcomm Cloud AI 100 Ultra**:

| Component | NVIDIA GPU | Qualcomm Cloud AI 100 |
|-----------|-----------|----------------------|
| **Resource request** | `nvidia.com/gpu: 1` | `qualcomm.com/qaic: 1` |
| **Container image** | `ollama/ollama` (or custom) | `ghcr.io/quic/cloud_ai_inference_ubuntu24:1.20.6.0` |
| **Inference engine** | Ollama / vLLM (CUDA) | vLLM (QAIC backend) |
| **Embedding model** | `mistral` via Ollama | Models compiled for QAIC via `/opt/vllm-env/` |

To run the Milvus RAG pipeline with a Qualcomm backend: (1) Deploy a Qualcomm inference pod using `yamls/qaic-inference.yaml` (Part 3). (2) Serve an embedding model via the vLLM OpenAI-compatible API on the QAIC device. (3) Point your embedding client at the Qualcomm pod's service endpoint. Milvus storage and retrieval stay the same. See the [Cloud AI SDK Kubernetes docs](https://quic.github.io/cloud-ai-sdk-pages/latest/Getting-Started/Installation/vLLM/vLLM/index.html) for Qualcomm deployment details.


This example uses a small set of sample documents to demonstrate Milvus vector storage and retrieval and RAG with the Ollama LLM.

## End

**Please make sure you did not leave any running pods. Jobs and associated completed pods are OK.**

**Join NRP & hands-on support:** Use these links to [get started with NRP](https://nrp.ai/documentation/userdocs/start/getting-started/) and to [join NRP's Matrix chat](https://nrp.ai/contact/) for hands-on support during the tutorial.

**Related documentation:** [NRP-managed vector database (Milvus)](https://nrp.ai/documentation/userdocs/vector-database/) · [NRP-managed LLMs](https://nrp.ai/documentation/userdocs/ai/llm-managed/) · [Qualcomm Cloud AI 100](https://nrp.ai/documentation/userdocs/qaic/)